## 1. Setup

In [ ]:
!pip install -q earthengine-api geemap scikit-learn pandas matplotlib


In [ ]:
import ee

# Replace with your own Google Cloud project ID that is registered for Earth Engine
PROJECT_ID = "your-gcp-project-id"

ee.Authenticate()
ee.Initialize(project=PROJECT_ID)

In [ ]:
print(ee.String('Earth Engine is working').getInfo())

## 2. Study area: Rawalpindi District

In [ ]:

# ============================================================
# GET THE ACTUAL RAWALPINDI DISTRICT BOUNDARY
# ============================================================

import ee
import geemap
from IPython.display import display

# Load Pakistan level 1 and level 2 boundaries to find the correct match
level1 = ee.FeatureCollection('FAO/GAUL/2015/level1')
level2 = ee.FeatureCollection('FAO/GAUL/2015/level2')

# Search in Level 1 (Divisions/Provinces) and Level 2 (Districts)
pakistan_l1 = level1.filter(ee.Filter.eq('ADM0_NAME', 'Pakistan'))
pakistan_l2 = level2.filter(ee.Filter.eq('ADM0_NAME', 'Pakistan'))

# Filter features containing 'Rawalpindi' (case-insensitive)
match_l1 = pakistan_l1.filter(ee.Filter.stringContains('ADM1_NAME', 'Rawalpindi'))
match_l2 = pakistan_l2.filter(ee.Filter.stringContains('ADM2_NAME', 'Rawalpindi'))

# Print found names
names_l1 = match_l1.aggregate_array('ADM1_NAME').getInfo()
names_l2 = match_l2.aggregate_array('ADM2_NAME').getInfo()

print("Matching Level 1 names:", names_l1)
print("Matching Level 2 names:", names_l2)

# Select the appropriate boundary based on search results
if len(names_l2) > 0:
    rawalpindi_fc = pakistan_l2.filter(ee.Filter.eq('ADM2_NAME', names_l2[0]))
elif len(names_l1) > 0:
    rawalpindi_fc = pakistan_l1.filter(ee.Filter.eq('ADM1_NAME', names_l1[0]))
else:
    # Fallback to a simplified bounding box if FAO GAUL has no match
    print("No GAUL match found, using fallback bounding box.")
    rawalpindi_fc = ee.FeatureCollection([ee.Feature(ee.Geometry.Rectangle([72.9, 33.45, 73.3, 33.8]))])

aoi = rawalpindi_fc.geometry()
print("Boundary loaded successfully.")

# Display boundary for verification
boundary_map = geemap.Map(basemap='Esri.WorldImagery')
boundary_map.centerObject(rawalpindi_fc, 10)
boundary_map.addLayer(
    rawalpindi_fc.style(
        color='FFFFFF',
        fillColor='00000000',
        width=3
    ),
    {},
    'Rawalpindi Boundary'
)
display(boundary_map)

## 3. Preliminary classification (superseded)

In [ ]:
def mask_s2(img):
    scl = img.select('SCL')
    mask = scl.neq(3).And(scl.neq(8)).And(scl.neq(9)).And(scl.neq(10))
    return img.updateMask(mask).divide(10000).copyProperties(img, ['system:time_start'])

s2 = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
      .filterBounds(aoi)
      .filterDate('2021-01-01', '2021-12-31')
      .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 30))
      .map(mask_s2))

composite = s2.median().clip(aoi)

bands = ['B2', 'B3', 'B4', 'B8', 'B11', 'B12']
ndvi = composite.normalizedDifference(['B8', 'B4']).rename('NDVI')
ndwi = composite.normalizedDifference(['B3', 'B8']).rename('NDWI')
ndbi = composite.normalizedDifference(['B11', 'B8']).rename('NDBI')

features_img = composite.select(bands).addBands([ndvi, ndwi, ndbi])
feature_names = bands + ['NDVI', 'NDWI', 'NDBI']

Map = geemap.Map(center=[33.62, 73.1], zoom=10, basemap='HYBRID')
Map.addLayer(composite, {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 0.3}, 'Sentinel-2 RGB 2021')
Map
worldcover = ee.ImageCollection('ESA/WorldCover/v200').first().select('Map')
label = worldcover.remap([10, 20, 30, 40, 50, 60, 80],
                         [1, 1, 1, 1, 2, 3, 0]).rename('label')

samples = features_img.addBands(label).stratifiedSample(
    numPoints=500, classBand='label', region=aoi, scale=10, seed=42, geometries=False)

df = geemap.ee_to_df(samples)
print(df.shape)
print(df['label'].value_counts())
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, cohen_kappa_score, classification_report, ConfusionMatrixDisplay

classes = [0, 1, 2, 3]
names = ['Water', 'Vegetation', 'Built-up', 'Bare land']

classifier = ee.Classifier.smileRandomForest(200, seed=42).train(
    features=samples, classProperty='label', inputProperties=feature_names)
classified = features_img.classify(classifier)

palette = ['0000FF', '00A000', 'FF0000', 'C2B280']
Map.addLayer(classified, {'min': 0, 'max': 3, 'palette': palette}, 'Land cover')
Map.add_legend(title='Land cover', keys=names, colors=['#' + c for c in palette])
Map

## 4. Spatial-block-validated classification



In [ ]:
# ============================================================
# MACHINE LEARNING-BASED LAND COVER CLASSIFICATION
# Study Area: Rawalpindi, Pakistan
# Satellite: Sentinel-2 SR Harmonized (2021)
# Reference labels: ESA WorldCover 2021
# Models: Scikit-learn Random Forest + Google Earth Engine RF
# ============================================================


# 1. IMPORT LIBRARIES
# ------------------------------------------------------------

import ee
import geemap
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (
    accuracy_score,
    cohen_kappa_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score
)

warnings.filterwarnings("ignore")


# 2. INITIALIZE GOOGLE EARTH ENGINE
# ------------------------------------------------------------

try:
    ee.Initialize(project=PROJECT_ID)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=PROJECT_ID)

print("Google Earth Engine initialized successfully.")


# The study-area geometry `aoi` is created in the boundary cell above.

if 'aoi' not in globals():
    raise ValueError(
        "AOI is missing. Run the study-area boundary cell first."
    )


# 3. DEFINE STUDY PARAMETERS
# ------------------------------------------------------------

START_DATE = '2021-01-01'
END_DATE = '2022-01-01'

RANDOM_SEED = 42
NUMBER_OF_TREES = 200

# Use a 20-metre analysis grid because Sentinel-2 B11 and B12
# have a native spatial resolution of 20 metres.

ANALYSIS_SCALE = 20

CLASS_VALUES = [0, 1, 2, 3]

CLASS_NAMES = [
    'Water',
    'Vegetated land',
    'Built-up',
    'Bare/sparse land'
]

PALETTE = [
    '0000FF',  # Water
    '00A000',  # Vegetated land
    'FF0000',  # Built-up
    'C2B280'   # Bare/sparse land
]

FEATURE_NAMES = [
    'B2', 'B3', 'B4', 'B8', 'B11', 'B12',
    'NDVI', 'NDWI', 'NDBI'
]


# 4. SENTINEL-2 CLOUD AND SHADOW MASKING
# ------------------------------------------------------------

def mask_s2(image):

    scl = image.select('SCL')

    # Exclude:
    # 0 = No data
    # 1 = Saturated/defective
    # 3 = Cloud shadow
    # 8 = Medium probability cloud
    # 9 = High probability cloud
    # 10 = Cirrus
    # 11 = Snow/ice

    mask = (
        scl.neq(0)
        .And(scl.neq(1))
        .And(scl.neq(3))
        .And(scl.neq(8))
        .And(scl.neq(9))
        .And(scl.neq(10))
        .And(scl.neq(11))
    )

    return (
        image.updateMask(mask)
        .divide(10000)
        .copyProperties(image, ['system:time_start'])
    )


# 5. LOAD SENTINEL-2 SATELLITE IMAGERY
# ------------------------------------------------------------

s2 = (
    ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    .filterBounds(aoi)
    .filterDate(START_DATE, END_DATE)
    .filter(
        ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 30)
    )
    .map(mask_s2)
)

image_count = s2.size().getInfo()

print("Number of Sentinel-2 images:", image_count)

if image_count == 0:
    raise ValueError(
        "No Sentinel-2 images found. Check your AOI and dates."
    )

# Median composite

composite = s2.median().clip(aoi)


# 6. CALCULATE SPECTRAL INDICES
# ------------------------------------------------------------

# Sentinel-2 spectral bands

bands = [
    'B2', 'B3', 'B4', 'B8', 'B11', 'B12'
]

# Vegetation index

ndvi = (
    composite.normalizedDifference(['B8', 'B4'])
    .rename('NDVI')
)

# Water index

ndwi = (
    composite.normalizedDifference(['B3', 'B8'])
    .rename('NDWI')
)

# Built-up index

ndbi = (
    composite.normalizedDifference(['B11', 'B8'])
    .rename('NDBI')
)

# Combine all predictor variables

features_img = (
    composite.select(bands)
    .addBands([ndvi, ndwi, ndbi])
)

print("Spectral bands and indices generated successfully.")


# 7. LOAD ESA WORLDCOVER REFERENCE LABELS
# ------------------------------------------------------------

worldcover = (
    ee.ImageCollection('ESA/WorldCover/v200')
    .first()
    .select('Map')
)

# Reclassify ESA WorldCover:
#
# 80 = Permanent water        -> 0
# 10 = Tree cover             -> 1
# 20 = Shrubland              -> 1
# 30 = Grassland              -> 1
# 40 = Cropland               -> 1
# 50 = Built-up               -> 2
# 60 = Bare/sparse vegetation -> 3
#
# 70, 90, 95 and 100 are excluded.

label = (
    worldcover.remap(
        [80, 10, 20, 30, 40, 50, 60],
        [0,  1,  1,  1,  1,  2,  3]
    )
    .rename('label')
)

# Mask out areas not included in the four classes.

label = label.updateMask(
    label.gte(0).And(label.lte(3))
)

print("ESA WorldCover reference classes prepared.")


# 8. GENERATE STRATIFIED TRAINING SAMPLES
# ------------------------------------------------------------

training_image = features_img.addBands(label)

# Sample 500 reference pixels per class.
# Geometries are retained for spatial block validation.

samples = training_image.stratifiedSample(
    numPoints=500,
    classBand='label',
    region=aoi,
    scale=ANALYSIS_SCALE,
    seed=RANDOM_SEED,
    geometries=True,
    dropNulls=True,
    tileScale=4
)

# Check sample count

print("Total sample count:", samples.size().getInfo())


# 9. CREATE SPATIAL BLOCK IDENTIFIERS
# ------------------------------------------------------------

# Rawalpindi is located in UTM Zone 43N.
# Divide the study area into approximately 2-km blocks.
# Samples in the same block will remain in the same
# training or testing partition.

projection = ee.Projection('EPSG:32643')

def add_spatial_block(feature):

    point = (
        feature.geometry()
        .centroid(1)
        .transform(projection, 1)
    )

    coordinates = point.coordinates()

    x = ee.Number(coordinates.get(0))
    y = ee.Number(coordinates.get(1))

    block_x = x.divide(2000).floor().format()
    block_y = y.divide(2000).floor().format()

    block_id = block_x.cat('_').cat(block_y)

    return feature.set('block_id', block_id)


samples = samples.map(add_spatial_block)

# Convert Earth Engine samples into a Pandas DataFrame.

df = geemap.ee_to_df(samples)

# Keep only the required variables.

required_columns = FEATURE_NAMES + [
    'label', 'block_id'
]

df = df[required_columns].copy()

# Remove invalid records.

df = df.replace([np.inf, -np.inf], np.nan)
df = df.dropna()

df['label'] = df['label'].astype(int)

print("\nSample dataset dimensions:", df.shape)

print("\nClass distribution:")
print(
    df['label']
    .value_counts()
    .sort_index()
    .rename(index=dict(zip(CLASS_VALUES, CLASS_NAMES)))
)


# 10. TRAINING AND TESTING PARTITION
# ------------------------------------------------------------

# GroupShuffleSplit keeps entire spatial blocks together.
# This reduces the risk of spatial leakage.

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.30,
    random_state=RANDOM_SEED
)

train_indices, test_indices = next(
    splitter.split(
        df[FEATURE_NAMES],
        df['label'],
        groups=df['block_id']
    )
)

train_df = df.iloc[train_indices].copy()
test_df = df.iloc[test_indices].copy()

X_train = train_df[FEATURE_NAMES]
X_test = test_df[FEATURE_NAMES]

y_train = train_df['label']
y_test = test_df['label']

print("\nTraining samples:", len(train_df))
print("Testing samples:", len(test_df))

print("\nTraining class distribution:")
print(train_df['label'].value_counts().sort_index())

print("\nTesting class distribution:")
print(test_df['label'].value_counts().sort_index())

print(
    "\nNumber of training spatial blocks:",
    train_df['block_id'].nunique()
)

print(
    "Number of testing spatial blocks:",
    test_df['block_id'].nunique()
)

if set(y_train.unique()) != set(CLASS_VALUES):
    raise ValueError(
        "One or more classes are missing from training data. "
        "Increase sampling or adjust the spatial block size."
    )

if set(y_test.unique()) != set(CLASS_VALUES):
    raise ValueError(
        "One or more classes are missing from testing data. "
        "Increase sampling or adjust the spatial block size."
    )


# 11. TRAIN SCIKIT-LEARN RANDOM FOREST
# ------------------------------------------------------------

rf_model = RandomForestClassifier(
    n_estimators=NUMBER_OF_TREES,
    max_features='sqrt',
    min_samples_leaf=1,
    random_state=RANDOM_SEED,
    n_jobs=-1,
    class_weight='balanced'
)

rf_model.fit(X_train, y_train)

print("\nScikit-learn Random Forest trained successfully.")


# 12. MODEL PREDICTION
# ------------------------------------------------------------

predictions = rf_model.predict(X_test)


# 13. MODEL ACCURACY ASSESSMENT
# ------------------------------------------------------------

overall_accuracy = accuracy_score(
    y_test,
    predictions
)

kappa = cohen_kappa_score(
    y_test,
    predictions
)

weighted_f1 = f1_score(
    y_test,
    predictions,
    average='weighted'
)

macro_f1 = f1_score(
    y_test,
    predictions,
    average='macro'
)

print("\n====================================")
print("MODEL PERFORMANCE")
print("====================================")

print("Overall Accuracy:", round(overall_accuracy, 4))
print("Cohen's Kappa:", round(kappa, 4))
print("Weighted F1-score:", round(weighted_f1, 4))
print("Macro F1-score:", round(macro_f1, 4))

print("\nClassification Report:\n")

print(
    classification_report(
        y_test,
        predictions,
        labels=CLASS_VALUES,
        target_names=CLASS_NAMES,
        zero_division=0
    )
)


# 14. CONFUSION MATRIX
# ------------------------------------------------------------

cm = confusion_matrix(
    y_test,
    predictions,
    labels=CLASS_VALUES
)

fig, ax = plt.subplots(figsize=(8, 6))

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=CLASS_NAMES
)

disp.plot(
    ax=ax,
    cmap='Blues',
    values_format='d',
    colorbar=False
)

plt.title(
    'Random Forest Confusion Matrix\n'
    'Spatial Block Validation'
)

plt.xticks(rotation=25)
plt.tight_layout()

plt.savefig(
    'confusion_matrix.png',
    dpi=300,
    bbox_inches='tight'
)

plt.show()


# 15. FEATURE IMPORTANCE
# ------------------------------------------------------------

importance = pd.Series(
    rf_model.feature_importances_,
    index=FEATURE_NAMES
).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(9, 6))

importance.plot(
    kind='barh',
    ax=ax
)

ax.set_title(
    'Random Forest Feature Importance'
)

ax.set_xlabel('Importance Score')
ax.set_ylabel('Predictor Variables')

plt.tight_layout()

plt.savefig(
    'feature_importance.png',
    dpi=300,
    bbox_inches='tight'
)

plt.show()


# 16. SAVE CLASSIFICATION REPORT
# ------------------------------------------------------------

report = classification_report(
    y_test,
    predictions,
    labels=CLASS_VALUES,
    target_names=CLASS_NAMES,
    output_dict=True,
    zero_division=0
)

report_df = pd.DataFrame(report).transpose()

report_df.to_csv(
    'classification_report.csv'
)

# Save model metrics

metrics = pd.DataFrame({
    'Metric': [
        'Overall Accuracy',
        "Cohen's Kappa",
        'Weighted F1-score',
        'Macro F1-score'
    ],
    'Value': [
        overall_accuracy,
        kappa,
        weighted_f1,
        macro_f1
    ]
})

metrics.to_csv(
    'model_metrics.csv',
    index=False
)


# 17. TRAIN GOOGLE EARTH ENGINE RANDOM FOREST
# ------------------------------------------------------------

# Use exactly the same spatial training and testing blocks
# as the Scikit-learn model.

train_block_ids = (
    train_df['block_id']
    .drop_duplicates()
    .tolist()
)

test_block_ids = (
    test_df['block_id']
    .drop_duplicates()
    .tolist()
)

train_samples = samples.filter(
    ee.Filter.inList('block_id', train_block_ids)
)

test_samples = samples.filter(
    ee.Filter.inList('block_id', test_block_ids)
)

# Train Earth Engine Random Forest

ee_classifier = ee.Classifier.smileRandomForest(
    numberOfTrees=NUMBER_OF_TREES,
    seed=RANDOM_SEED
).train(
    features=train_samples,
    classProperty='label',
    inputProperties=FEATURE_NAMES
)

print("\nEarth Engine Random Forest trained.")


# 18. VALIDATE EARTH ENGINE CLASSIFIER
# ------------------------------------------------------------

test_predictions = test_samples.classify(
    ee_classifier
)

ee_confusion_matrix = test_predictions.errorMatrix(
    'label',
    'classification',
    CLASS_VALUES
)

ee_accuracy = ee_confusion_matrix.accuracy().getInfo()

ee_kappa = ee_confusion_matrix.kappa().getInfo()

print("\n====================================")
print("EARTH ENGINE MODEL VALIDATION")
print("====================================")

print("Overall Accuracy:", round(ee_accuracy, 4))
print("Kappa:", round(ee_kappa, 4))

print("\nEarth Engine Confusion Matrix:")

print(
    ee_confusion_matrix
    .getInfo()
)


# 19. GENERATE FINAL LAND COVER MAP
# ------------------------------------------------------------

classified = (
    features_img
    .classify(ee_classifier)
    .clip(aoi)
)

print("\nFinal land-cover classification generated.")



# ============================================================
# 20. LAND COVER CLASSIFICATION MAP
# RAWALPINDI, PAKISTAN | 2021
# ============================================================

import ee
import geemap
from IPython.display import display

# 1. Create a new map using Esri satellite imagery
# This avoids the blocked OpenStreetMap tile server.

Map = geemap.Map(
    basemap='Esri.WorldImagery',
    center=[33.62, 73.10],
    zoom=10
)

# 2. Define clear, strong classification colours

PALETTE = [
    '0000FF',  # Water - Blue
    '008000',  # Vegetated land - Green
    'FF0000',  # Built-up - Red
    'D2B48C'   # Bare/sparse land - Tan
]

# 3. Add final classified image
# Full opacity makes the four classes easier to distinguish.

Map.addLayer(
    classified.clip(aoi),
    {
        'min': 0,
        'max': 3,
        'palette': PALETTE
    },
    'Land Cover Classification 2021',
    True,
    1.0
)

# 4. Add study area boundary

boundary = ee.Image().byte().paint(
    featureCollection=ee.FeatureCollection(
        aoi
    ),
    color=1,
    width=2
)

Map.addLayer(
    boundary,
    {'palette': ['FFFFFF']},
    'Study Area Boundary',
    True
)

# 5. Add Sentinel-2 RGB as an optional background layer

Map.addLayer(
    composite,
    {
        'bands': ['B4', 'B3', 'B2'],
        'min': 0.02,
        'max': 0.30,
        'gamma': 1.2
    },
    'Sentinel-2 RGB 2021',
    False
)

# 6. Add NDVI as an optional layer

Map.addLayer(
    ndvi,
    {
        'min': -0.5,
        'max': 0.8,
        'palette': ['brown', 'yellow', 'green']
    },
    'NDVI 2021',
    False
)

# 7. Add a clear classification legend

Map.add_legend(
    title='LAND COVER | 2021',
    keys=[
        'Water',
        'Vegetated land',
        'Built-up',
        'Bare/sparse land'
    ],
    colors=[
        '#0000FF',
        '#008000',
        '#FF0000',
        '#D2B48C'
    ]
)

# 8. Set the map extent to Rawalpindi study area

Map.centerObject(aoi, 10)

# 9. Add layer selection controls

Map.addLayerControl()

# 10. Display the final interactive map

display(Map)


# 21. FINAL PROJECT SUMMARY
# ------------------------------------------------------------

print("\n====================================")
print("PROJECT COMPLETED")
print("\n====================================")

print("Study Area: Rawalpindi, Pakistan")
print("Satellite: Sentinel-2 SR Harmonized")
print("Year: 2021")
print("Analysis Resolution:", ANALYSIS_SCALE, "metres")
print("Classification Model: Random Forest")
print("Reference Dataset: ESA WorldCover 2021")

print("\nClasses:")

for class_id, class_name in zip(
    CLASS_VALUES,
    CLASS_NAMES
):
    print(class_id, ":", class_name)

print("\nScikit-learn Overall Accuracy:",
      round(overall_accuracy, 4))

print("Earth Engine Overall Accuracy:",
      round(ee_accuracy, 4))

print("\nOutput files created:")
print("- confusion_matrix.png")
print("- feature_importance.png")
print("- classification_report.csv")

print("\nProject workflow completed successfully.")

## 5. Clean interactive map

In [ ]:

# ============================================================
# CLEAN INTERACTIVE LAND COVER MAP
# ============================================================

import geemap
from IPython.display import display

# Create a standard map
Map = geemap.Map()

# Clear the default background basemap layers
Map.clear_layers()

Map.centerObject(rawalpindi_fc, 10)

# Add classified image
Map.addLayer(
    classified.clip(aoi),
    {
        'min': 0,
        'max': 3,
        'palette': [
            '0000FF',  # Water
            '008000',  # Vegetated land
            'FF0000',  # Built-up
            'D2B48C'   # Bare/sparse land
        ]
    },
    'Land Cover Classification 2021'
)

# Add district boundary
Map.addLayer(
    rawalpindi_fc.style(
        color='000000',
        fillColor='00000000',
        width=3
    ),
    {},
    'Rawalpindi District Boundary'
)

# Add legend
Map.add_legend(
    title='LAND COVER CLASSIFICATION',
    keys=[
        'Water',
        'Vegetated land',
        'Built-up',
        'Bare/sparse land'
    ],
    colors=[
        '#0000FF',
        '#008000',
        '#FF0000',
        '#D2B48C'
    ]
)

display(Map)

## 6. Save evaluation tables

In [ ]:

# ============================================================
# SAVE ALL MODEL ACCURACY AND EVALUATION RESULTS
# ============================================================

import pandas as pd
import numpy as np

from sklearn.metrics import classification_report

# 1. Overall model performance

overall_metrics = pd.DataFrame({
    'Metric': [
        'Overall Accuracy',
        "Cohen's Kappa",
        'Weighted F1-score',
        'Macro F1-score'
    ],
    'Value': [
        accuracy_score(y_test, predictions),
        cohen_kappa_score(y_test, predictions),
        f1_score(y_test, predictions, average='weighted'),
        f1_score(y_test, predictions, average='macro')
    ]
})

overall_metrics.to_csv(
    'Overall_Accuracy_Metrics.csv',
    index=False
)

# 2. Class-wise accuracy assessment

class_report = classification_report(
    y_test,
    predictions,
    labels=CLASS_VALUES,
    target_names=CLASS_NAMES,
    output_dict=True,
    zero_division=0
)

class_metrics = pd.DataFrame(class_report).transpose()

class_metrics.to_csv(
    'Class_Wise_Accuracy_Report.csv'
)

# 3. Confusion matrix

cm = confusion_matrix(
    y_test,
    predictions,
    labels=CLASS_VALUES
)

confusion_df = pd.DataFrame(
    cm,
    index=CLASS_NAMES,
    columns=CLASS_NAMES
)

confusion_df.index.name = 'Actual'
confusion_df.columns.name = 'Predicted'

confusion_df.to_csv(
    'Confusion_Matrix.csv'
)

# 4. Feature importance

feature_importance = pd.DataFrame({
    'Feature': FEATURE_NAMES,
    'Importance': rf_model.feature_importances_
}).sort_values(
    by='Importance',
    ascending=False
)

feature_importance.to_csv(
    'Feature_Importance.csv',
    index=False
)

# 5. Save Earth Engine validation metrics

ee_metrics = pd.DataFrame({
    'Metric': [
        'Overall Accuracy',
        "Cohen's Kappa"
    ],
    'Value': [
        ee_accuracy,
        ee_kappa
    ]
})

ee_metrics.to_csv(
    'Earth_Engine_Accuracy.csv',
    index=False
)

# 6. Display saved results

print("\nOVERALL ACCURACY")
display(overall_metrics)

print("\nCLASS-WISE PERFORMANCE")
display(class_metrics)

print("\nCONFUSION MATRIX")
display(confusion_df)

print("\nFEATURE IMPORTANCE")
display(feature_importance)

print("\nAll evaluation files saved successfully.")


## 7. Export the classified raster to Google Drive

In [ ]:
# Export the classified image to Google Drive as a GeoTIFF.
# (Reconstructed cell: the export step was missing from the original notebook.
#  Settings follow the map footer: 20 m resolution, EPSG:32643.)
export_task = ee.batch.Export.image.toDrive(
    image=classified.toByte(),
    description='Rawalpindi_LULC_RF_2021',
    folder='Rawalpindi_Geospatial_Project',
    fileNamePrefix='Rawalpindi_LULC_RF_2021',
    region=aoi,
    scale=ANALYSIS_SCALE,
    crs='EPSG:32643',
    maxPixels=1e13,
    fileFormat='GeoTIFF'
)
export_task.start()
print("Export started. Run the next cell to check its state.")

In [ ]:
print(export_task.status()["state"])   # wait for COMPLETED before continuing

Mount Google Drive and define the working folder.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
folder = '/content/drive/MyDrive/Rawalpindi_Geospatial_Project'
os.makedirs(folder, exist_ok=True)


## 8. Inspect the exported raster

In [ ]:
import rasterio

tif_path = folder + '/Rawalpindi_LULC_RF_2021.tif'

with rasterio.open(tif_path) as src:
    print("CRS:", src.crs)
    print("Raster dimensions:", src.width, "x", src.height)
    print("Resolution:", src.res)
    print("Number of bands:", src.count)
    print("Data type:", src.dtypes)

In [ ]:
import rasterio
import numpy as np

with rasterio.open(tif_path) as src:
    data = src.read(1, masked=True)

    print("Unique class values:")
    print(np.unique(data.compressed()))

    print("NoData value:", src.nodata)
    print("Valid pixels:", data.count())

## 9. Final map with inset, north arrow and scale bar

In [ ]:

import ee
import geemap
import geopandas as gpd

# Retrieve Rawalpindi District boundary
district_fc = ee.FeatureCollection(
    'FAO/GAUL/2015/level2'
).filter(
    ee.Filter.And(
        ee.Filter.eq('ADM0_NAME', 'Pakistan'),
        ee.Filter.eq('ADM2_NAME', 'Rawalpindi District')
    )
)

# Verify that the district was found
print("District features:", district_fc.size().getInfo())

district_gdf = geemap.ee_to_gdf(district_fc)
district_gdf = district_gdf.to_crs('EPSG:32643')

# Retrieve Pakistan boundary for inset map
pakistan_fc = ee.FeatureCollection(
    'FAO/GAUL/2015/level0'
).filter(
    ee.Filter.eq('ADM0_NAME', 'Pakistan')
)

print("Pakistan features:", pakistan_fc.size().getInfo())

pakistan_gdf = geemap.ee_to_gdf(pakistan_fc)

print("District boundary prepared successfully.")


In [ ]:

import os
import numpy as np
import rasterio
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from rasterio.features import geometry_mask
from matplotlib.colors import ListedColormap, BoundaryNorm
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from pyproj import Transformer

# --------------------------------------------------
# 1. File paths

import os

output_folder = '/content/drive/MyDrive/Rawalpindi_Geospatial_Project'
os.makedirs(output_folder, exist_ok=True)

tif_path = os.path.join(
    output_folder,
    'Rawalpindi_LULC_RF_2021.tif'
)

print("Output folder:", output_folder)

# --------------------------------------------------
# 2. Read raster and mask outside district boundary
# --------------------------------------------------

with rasterio.open(tif_path) as src:

    raster = src.read(1)
    transform = src.transform
    raster_crs = src.crs
    bounds = src.bounds

    district_geom = district_gdf.to_crs(raster_crs).geometry

    inside_mask = geometry_mask(
        district_geom,
        transform=transform,
        invert=True,
        out_shape=raster.shape
    )

    raster = np.where(inside_mask, raster, np.nan)

# --------------------------------------------------
# 3. Class colours and labels
# --------------------------------------------------

colors = [
    '#3186C8',  # Water
    '#39964A',  # Vegetation
    '#D94A3A',  # Built-up
    '#D9B879'   # Bare land
]

class_names = [
    'Water',
    'Vegetation',
    'Built-up',
    'Bare land'
]

cmap = ListedColormap(colors)
norm = BoundaryNorm([-0.5, 0.5, 1.5, 2.5, 3.5], cmap.N)

# --------------------------------------------------
# 4. Create map layout
# --------------------------------------------------

fig = plt.figure(figsize=(14, 12))
fig.patch.set_facecolor('white')

# Main Rawalpindi map: reserve space on the right
ax = fig.add_axes([0.08, 0.12, 0.65, 0.76])
ax.set_facecolor('white')

fig.patch.set_facecolor('white')
ax.set_facecolor('white')

extent = [
    bounds.left,
    bounds.right,
    bounds.bottom,
    bounds.top
]

im = ax.imshow(
    raster,
    extent=extent,
    origin='upper',
    cmap=cmap,
    norm=norm,
    interpolation='nearest'
)

# District boundary
district_gdf.boundary.plot(
    ax=ax,
    color='black',
    linewidth=1.4,
    zorder=3
)

# --------------------------------------------------
# 5. Title
# --------------------------------------------------

fig.suptitle(
    'LAND USE / LAND COVER CLASSIFICATION',
    fontsize=19,
    fontweight='bold',
    y=0.96
)

ax.set_title(
    'Rawalpindi District, Punjab, Pakistan | 2021',
    fontsize=13,
    pad=12
)

# --------------------------------------------------
# 6. Legend
# --------------------------------------------------

legend_handles = [
    mpatches.Patch(
        color=colors[i],
        label=class_names[i]
    )
    for i in range(4)
]

ax.legend(
    handles=legend_handles,
    title='Land Cover Classes',
    loc='lower left',
    frameon=True,
    facecolor='white',
    edgecolor='black',
    fontsize=10,
    title_fontsize=11
)

# --------------------------------------------------
# 7. North arrow
# --------------------------------------------------

# Dedicated north arrow position
north_ax = fig.add_axes([0.81, 0.43, 0.10, 0.17])
north_ax.set_axis_off()

north_ax.annotate(
    'N',
    xy=(0.5, 0.92),
    xytext=(0.5, 0.20),
    xycoords='axes fraction',
    textcoords='axes fraction',
    ha='center',
    va='center',
    fontsize=17,
    fontweight='bold',
    arrowprops=dict(
        facecolor='black',
        edgecolor='black',
        width=3,
        headwidth=12
    )
)

# --------------------------------------------------
# 8. Scale bar - bottom right of main map
# --------------------------------------------------

# Position relative to the main map axes
x_start = 0.68
y_position = 0.055
bar_width = 0.20

# Draw a 5 km scale bar
ax.plot(
    [x_start, x_start + bar_width],
    [y_position, y_position],
    transform=ax.transAxes,
    color='black',
    linewidth=4,
    solid_capstyle='butt',
    zorder=10
)

# End markers
for x in [x_start, x_start + bar_width]:
    ax.plot(
        [x, x],
        [y_position - 0.022, y_position + 0.022],
        transform=ax.transAxes,
        color='black',
        linewidth=1.5,
        zorder=10
    )

ax.text(
    x_start,
    y_position - 0.025,
    '0',
    transform=ax.transAxes,
    ha='center',
    va='top',
    fontsize=9
)

ax.text(
    x_start + bar_width,
    y_position - 0.025,
    '5 km',
    transform=ax.transAxes,
    ha='center',
    va='top',
    fontsize=9
)

# --------------------------------------------------
# 9. Coordinate grid
# --------------------------------------------------

ax.ticklabel_format(style='plain', axis='both')

ax.set_xlabel('Easting (metres)', fontsize=10)
ax.set_ylabel('Northing (metres)', fontsize=10)

ax.grid(
    True,
    linestyle='--',
    linewidth=0.5,
    alpha=0.5
)

# --------------------------------------------------
# 10. Pakistan location inset
# --------------------------------------------------

# Pakistan locator map in a separate fixed position
inset = fig.add_axes([0.76, 0.68, 0.19, 0.21])

pakistan_plot = pakistan_gdf.to_crs('EPSG:4326')
district_wgs = district_gdf.to_crs('EPSG:4326')

pakistan_plot.plot(
    ax=inset,
    color='#E8E8E8',
    edgecolor='black',
    linewidth=0.8
)

# Highlight Rawalpindi
district_center = district_wgs.geometry.union_all().centroid

inset.scatter(
    district_center.x,
    district_center.y,
    color='red',
    marker='*',
    s=100,
    zorder=5
)

# Fix the inset extent to show Pakistan, not just Rawalpindi
inset.set_xlim(60, 78)
inset.set_ylim(23, 38)
inset.set_aspect('equal')

inset.set_title(
    'Study Area in Pakistan',
    fontsize=10,
    fontweight='bold',
    pad=7
)

inset.set_axis_off()
# --------------------------------------------------
# 11. Source information
# --------------------------------------------------

fig.text(
    0.5,
    0.035,
    'Data: Sentinel-2 Surface Reflectance | Classification: Random Forest '
    '| Resolution: 20 m | CRS: EPSG:32643',
    ha='center',
    fontsize=8
)

fig.text(
    0.5,
    0.018,
    'Prepared using Python, Rasterio, GeoPandas and Google Earth Engine',
    ha='center',
    fontsize=8,
    style='italic'
)

# --------------------------------------------------
# 12. Save final map
# --------------------------------------------------

png_path = os.path.join(output_folder, 'Rawalpindi_Land_Cover_Map_2021.png')
pdf_path = os.path.join(output_folder, 'Rawalpindi_Land_Cover_Map_2021.pdf')

fig.savefig(png_path, dpi=300, bbox_inches='tight', facecolor='white')
fig.savefig(pdf_path, bbox_inches='tight', facecolor='white')

plt.show()

print('Map saved successfully:')
print(png_path)
print(pdf_path)
